### **spacy_text_classification : Exercise**


- In this exercise, you are going to classify whether a given text belongs to one of possible classes ['BUSINESS', 'SPORTS', 'CRIME'].

- you are going to use spacy for pre-processing the text, convert text to numbers and apply different classification algorithms.

In [1]:
#uncomment the below line and run this cell to install the large english model which is trained on wikipedia data

# !python -m spacy download en_core_web_lg

In [38]:
#import spacy and load the language model downloaded

import spacy
nlp = spacy.load("en_core_web_lg")


### **About Data: News Category Classifier**

Credits: https://www.kaggle.com/code/hengzheng/news-category-classifier-val-acc-0-65


- This data consists of two columns.
        - Text
        - Category
- Text are the description about a particular topic.
- Category determine which class the text belongs to.
- we have classes mainly of 'BUSINESS', 'SPORTS', 'CRIME' and comes under **Multi-class** classification Problem.

In [39]:
#import pandas library
import pandas as pd


#read the dataset "news_dataset.json" provided and load it into dataframe "df"
df = pd.read_json("news_dataset.json")



#print the shape of data
print(df.shape)


#print the top5 rows
print(df.head())


(7500, 2)
                                                text  category
0  Larry Nassar Blames His Victims, Says He 'Was ...     CRIME
1       Woman Beats Cancer, Dies Falling From Horse      CRIME
2  Vegas Taxpayers Could Spend A Record $750 Mill...    SPORTS
3  This Richard Sherman Interception Literally Sh...    SPORTS
4  7 Things That Could Totally Kill Weed Legaliza...  BUSINESS


In [40]:
#check the distribution of labels 

df['category'].value_counts()

category
CRIME       2500
SPORTS      2500
BUSINESS    2500
Name: count, dtype: int64

In [41]:
#Add the new column "label_num" which gives a unique number to each of these labels 

df['label_num'] = df['category'].map({
    'BUSINESS':0, 'CRIME':1, 'SPORTS':2
})

#check the results with top 5 rows
df.head()

,text,category,label_num
0,"Larry Nassar Blames His Victims, Says He 'Was ...",CRIME,1
1,"Woman Beats Cancer, Dies Falling From Horse",CRIME,1
2,Vegas Taxpayers Could Spend A Record $750 Mill...,SPORTS,2
3,This Richard Sherman Interception Literally Sh...,SPORTS,2
4,7 Things That Could Totally Kill Weed Legaliza...,BUSINESS,0


### **Preprocess the text**

In [42]:
#use this utility function to preprocess the text
#1. Remove the stop words
#2. Convert to base form using lemmatisation

def preprocess(text):
    doc = nlp(text)
    filtered_tokens = []
    for token in doc:
        if token.is_stop or token.is_punct:
            continue
        filtered_tokens.append(token.lemma_)
    return ' '.join(filtered_tokens)

In [43]:
#create a new column "preprocessed_text" which store the clean form of given text [use apply and lambda function]
df['preprocessed_text'] = df['text'].apply(lambda x: preprocess(x))


In [44]:
#print the top 5 rows
df.head()

,text,category,label_num,preprocessed_text
0,"Larry Nassar Blames His Victims, Says He 'Was ...",CRIME,1,Larry Nassar blame Victims say victimize newly...
1,"Woman Beats Cancer, Dies Falling From Horse",CRIME,1,woman Beats Cancer dies fall Horse
2,Vegas Taxpayers Could Spend A Record $750 Mill...,SPORTS,2,Vegas Taxpayers spend record $ 750 million New...
3,This Richard Sherman Interception Literally Sh...,SPORTS,2,Richard Sherman Interception literally shake W...
4,7 Things That Could Totally Kill Weed Legaliza...,BUSINESS,0,7 thing totally kill weed legalization Buzz


### **Get the spacy embeddings for each preprocessed text**

In [45]:
#create a new column "vector" that store the vector representation of each pre-processed text

df['vector'] = df['preprocessed_text'].apply(lambda x: nlp(x).vector)


In [46]:
#print the top 5 rows
df.head()

,text,category,label_num,preprocessed_text,vector
0,"Larry Nassar Blames His Victims, Says He 'Was ...",CRIME,1,Larry Nassar blame Victims say victimize newly...,"[-0.3472573, 0.021758832, -0.2137525, -0.01718..."
1,"Woman Beats Cancer, Dies Falling From Horse",CRIME,1,woman Beats Cancer dies fall Horse,"[-0.16791849, 0.43708333, 0.035527337, 0.01661..."
2,Vegas Taxpayers Could Spend A Record $750 Mill...,SPORTS,2,Vegas Taxpayers spend record $ 750 million New...,"[0.053351898, 0.08053064, -0.05101806, -0.1991..."
3,This Richard Sherman Interception Literally Sh...,SPORTS,2,Richard Sherman Interception literally shake W...,"[-0.038867258, 0.28459162, 0.071352966, -0.045..."
4,7 Things That Could Totally Kill Weed Legaliza...,BUSINESS,0,7 thing totally kill weed legalization Buzz,"[-0.20180944, 0.11867001, 0.0036708585, -0.189..."


**Train-Test splitting**

In [47]:

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    df.vector,
    df.label_num,
    random_state=2022
)


**Reshape the X_train and X_test so as to fit for models**

In [48]:
# import numpy as np

import numpy as np

#reshapes the X_train and X_test using 'stack' function of numpy. Store the result in new variables "X_train_2d" and "X_test_2d"
X_train_2d = np.stack(X_train)
X_test_2d = np.stack(X_test)

**Attempt 1:**


- use spacy glove embeddings for text vectorization.

- use Decision Tree as the classifier.

- print the classification report.

In [49]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

#1. creating a Decision Tree model object

model = DecisionTreeClassifier()

#2. fit with all_train_embeddings and y_train

model.fit(X_train_2d, y_train)

#3. get the predictions for all_test_embeddings and store it in y_pred
y_pred = model.predict(X_test_2d)


#4. print the classfication report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.75      0.71      0.73       653
           1       0.77      0.78      0.77       609
           2       0.73      0.77      0.75       613

    accuracy                           0.75      1875
   macro avg       0.75      0.75      0.75      1875
weighted avg       0.75      0.75      0.75      1875



**Attempt 2:**


- use spacy glove embeddings for text vectorization.
- use MultinomialNB as the classifier after applying the MinMaxscaler.
- print the classification report.

In [50]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report



#doing scaling because Negative values will not pass into Naive Bayes models
scalar = MinMaxScaler()
scaled_x_train = scalar.fit_transform(X_train_2d)
scaled_x_test = scalar.transform(X_test_2d)


#1. creating a MultinomialNB model object 
model1 = MultinomialNB()


#2. fit with all_train_embeddings(scaled) and y_train
model1.fit(scaled_x_train, y_train)


#3. get the predictions for all_test_embeddings and store it in y_pred
y_pred = model1.predict(scaled_x_test)


#4. print the classfication report
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.89      0.89      0.89       653
           1       0.89      0.91      0.90       609
           2       0.90      0.88      0.89       613

    accuracy                           0.89      1875
   macro avg       0.89      0.89      0.89      1875
weighted avg       0.89      0.89      0.89      1875



**Attempt 3:**


- use spacy glove embeddings for text vectorization.
- use KNeighborsClassifier as the classifier after applying the MinMaxscaler.
- print the classification report.

In [51]:
from  sklearn.neighbors import KNeighborsClassifier


#1. creating a KNN model object
model = KNeighborsClassifier()


#2. fit with all_train_embeddings and y_train
model.fit(scaled_x_train, y_train)


#3. get the predictions for all_test_embeddings and store it in y_pred
y_pred = model.predict(scaled_x_test)


#4. print the classfication report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.88      0.92      0.90       653
           1       0.87      0.94      0.90       609
           2       0.96      0.84      0.89       613

    accuracy                           0.90      1875
   macro avg       0.90      0.90      0.90      1875
weighted avg       0.90      0.90      0.90      1875



**Attempt 4:**


- use spacy glove embeddings for text vectorization.
- use RandomForestClassifier as the classifier after applying the MinMaxscaler.
- print the classification report.

In [52]:
from sklearn.ensemble import RandomForestClassifier


#1. creating a Random Forest model object
model = RandomForestClassifier()


#2. fit with all_train_embeddings and y_train
model.fit(scaled_x_train, y_train)


#3. get the predictions for all_test_embeddings and store it in y_pred
y_pred = model.predict(scaled_x_test)


#4. print the classfication report
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.91      0.90      0.91       653
           1       0.89      0.91      0.90       609
           2       0.91      0.91      0.91       613

    accuracy                           0.90      1875
   macro avg       0.90      0.90      0.90      1875
weighted avg       0.90      0.90      0.90      1875



**Attempt 5:**


- use spacy glove embeddings for text vectorization.
- use GradientBoostingClassifier as the classifier after applying the MinMaxscaler.
- print the classification report.

In [53]:
from sklearn.ensemble import GradientBoostingClassifier


#1. creating a GradientBoosting model object
model = GradientBoostingClassifier()


#2. fit with all_train_embeddings and y_train
model.fit(scaled_x_train, y_train)


#3. get the predictions for all_test_embeddings and store it in y_pred
y_pred = model.predict(scaled_x_test)


#4. print the classfication report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.91      0.91      0.91       653
           1       0.90      0.90      0.90       609
           2       0.92      0.91      0.91       613

    accuracy                           0.91      1875
   macro avg       0.91      0.91      0.91      1875
weighted avg       0.91      0.91      0.91      1875



**Print the confusion Matrix with the best model got**

In [13]:
#finally print the confusion matrix for the best model: GradientBoostingClassifier

# from sklearn.metrics import confusion_matrix





## [**Solution**](./spacy_word_embeddings_solution.ipynb)